## **แบบฝึกหัดขั้นสูง: Multi-Function Calling และการจัดการข้อมูลด้วย Pydantic**


---

### **ข้อที่ 1: ระบบวางแผนการเดินทางอัจฉริยะ**

**สถานการณ์:** ผู้ใช้ต้องการวางแผนการเดินทางจากโตเกียว (Tokyo) ไปยังปารีส (Paris) และต้องการข้อมูลเปรียบเทียบ ทั้งเรื่องเวลาท้องถิ่นของทั้งสองเมือง, สภาพอากาศปัจจุบัน, และเมืองหลวงของประเทศนั้นๆ เพื่อยืนยันข้อมูล

**งานของคุณ:**
1.  เขียนโค้ดที่รับคำสั่งจากผู้ใช้ เช่น `"ฉันจะเดินทางจากโตเกียวไปปารีส ช่วยบอกเวลาท้องถิ่นและสภาพอากาศของทั้งสองเมืองหน่อย แล้วเมืองหลวงของญี่ปุ่นกับฝรั่งเศสคือที่ไหนนะ?"`
2.  คุณจะต้องสร้าง Mock Function ใหม่ชื่อ `get_weather(city: str)` ซึ่งจะคืนค่าเป็นข้อมูลสภาพอากาศจำลอง (เช่น `{"temperature": "15°C", "condition": "Cloudy"}`)
3.  เรียกใช้ฟังก์ชัน `get_current_time`, `get_capital_city`, และ `get_weather` สำหรับเมืองทั้งสอง
4.  ** (ส่วนที่ท้าทาย)** ออกแบบ Pydantic Model ชื่อ `TripPlannerResponse` เพื่อรวบรวมข้อมูลทั้งหมด โดยมีการแบ่งโครงสร้างข้อมูลของแต่ละเมืองอย่างชัดเจน

**Pydantic Model ที่คาดหวัง:**
```python
from pydantic import BaseModel
from typing import Dict

class CityInfo(BaseModel):
    local_time: str
    weather: Dict[str, str] # e.g., {"temperature": "15°C", "condition": "Cloudy"}
    capital_of: str

class TripPlannerResponse(BaseModel):
    origin: CityInfo
    destination: CityInfo
    trip_summary: str # สร้างข้อความสรุปสั้นๆ จากข้อมูลทั้งหมด
```
**คำใบ้:** โมเดล LLM อาจจะต้องเรียกฟังก์ชันเดียวกัน (เช่น `get_current_time`) แต่ด้วยพารามิเตอร์ที่ต่างกัน (คนละเมือง) ในคำสั่งเดียว คุณต้องจัดการผลลัพธ์เหล่านี้ให้ถูกต้องและจับคู่กับเมืองที่ถูกต้องใน Pydantic Model

In [32]:
import IPython
import sys

def clean_notebook():
    IPython.display.clear_output(wait=True)
    print("Notebook cleaned.")

# Install necessary libraries
!pip install openai python-dotenv -q

# Clean up the notebook
clean_notebook()

Notebook cleaned.


In [33]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [34]:
from pydantic import BaseModel, Field
from typing import Dict

class WeatherInfo(BaseModel):
    temperature: str
    condition: str

class CityInfo(BaseModel):
    local_time: str
    weather: WeatherInfo =Field(description="e.g., {'temperature': '15°C', 'condition': 'Cloudy'}")
    capital_city: str
    country: str

class TripPlannerResponse(BaseModel):
    origin: CityInfo =Field(description="ข้อมูลประเทศต้นทาง")
    destination: CityInfo =Field(description="ข้อมูลประเทศปลายทาง")
    trip_summary: str =Field(description="ข้อความสรุปสั้นๆ จากข้อมูลทั้งหมด")


In [35]:
import random
import datetime
import pytz
import json

# Mock function
def get_weather(city: str):
    """Gets random weather info for a city."""
    
    possible_conditions = ["Sunny", "Cloudy", "Rainy", "Stormy"]
    temperature = f"{random.randint(-5, 40)}°C"
    condition = random.choice(possible_conditions)

    weather_data = {
        "Tokyo": WeatherInfo(temperature=temperature, condition=condition),
        "Paris": WeatherInfo(temperature=temperature, condition=condition)
    }

    return weather_data.get(city, f"Weather info for {city} not found.")

# Mock function
def get_capital_city(country):
    """Gets the capital city of a country."""
    capitals = {"Japan": "Tokyo", "France": "Paris"}
    return capitals.get(country, f"Capital of {country} not found.")

# Mock function
def get_current_time(timezone):
    """Gets the current time in a specific timezone."""
    try:
        tz = pytz.timezone(timezone)
        current_time = datetime.datetime.now(tz)
        return f"The current time in {timezone} is {current_time.strftime('%H:%M:%S')}."
    except pytz.UnknownTimeZoneError:
        return f"Unknown timezone: {timezone}"

In [36]:
all_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Gets random weather info for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_capital_city",
            "description": "Get the capital city of a country.",
            "parameters": {
                "type": "object",
                "properties": {
                    "country": {"type": "string"}
                },
                "required": ["country"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current time in a specific timezone.",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {
                        "type": "string",
                        "description": "The timezone, e.g., 'France/Paris' or 'Asia/Tokyo'."
                    }
                },
                "required": ["timezone"]
            }
        }
    }
]




In [37]:
def execute_multi_function_call(user_query, tools):
    """A helper function to execute the full multi-function calling flow."""
    messages = [{"role": "user", "content": user_query}]
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    messages.append(response_message)  
    
    if response_message.tool_calls:
        available_functions = {
            "get_weather": get_weather,
            "get_capital_city": get_capital_city,
            "get_current_time": get_current_time,
        }
        
        
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions.get(function_name)
            
            if function_to_call:
                function_args = json.loads(tool_call.function.arguments)
                function_response = function_to_call(**function_args)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(function_response.model_dump() if isinstance(function_response, BaseModel) else function_response)
                })

        
        second_response = client.beta.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=messages,
        response_format=TripPlannerResponse,
        )
    
        return second_response.choices[0].message.parsed
        
    else:
        return response_message.content
    


In [ ]:
user_query = input("กรอกข้อมูลการเดินทางของคุณ: ") #รับค่าทางแป้นพิมพ์
final_response = execute_multi_function_call(user_query, all_tools)



In [45]:
for field, value in final_response.model_dump().items():
        if isinstance(value, list):
            print(f"{field}:")
            for idx, item in enumerate(value, 1):
                print(f"  item no. {idx}:")
                for subfield, subvalue in item.items():
                    print(f"    {subfield}: {subvalue}")
        else:
            print(f"{field}: {value}")

origin: {'local_time': '21:43', 'weather': {'temperature': '-4°C', 'condition': 'Rainy'}, 'capital_city': 'Tokyo', 'country': 'Japan'}
destination: {'local_time': '14:43', 'weather': {'temperature': '27°C', 'condition': 'Cloudy'}, 'capital_city': 'Paris', 'country': 'France'}
trip_summary: คุณจะเดินทางจากโตเกียว ประเทศญี่ปุ่น ไปยังปารีส ประเทศฝรั่งเศส ดังนี้:

### เมืองต้นทาง: โตเกียว, ประเทศญี่ปุ่น
- **เวลาท้องถิ่น**: 21:43
- **สภาพอากาศ**: อุณหภูมิ -4°C, ฝนตก
- **เมืองหลวง**: โตเกียว

### เมืองปลายทาง: ปารีส, ประเทศฝรั่งเศส
- **เวลาท้องถิ่น**: 14:43
- **สภาพอากาศ**: อุณหภูมิ 27°C, เมฆบางส่วน
- **เมืองหลวง**: ปารีส

ขอเดินทางปลอดภัยนะคะ!


### **ข้อที่ 2: ระบบสรุปข้อมูลเหตุการณ์สำคัญ**

**สถานการณ์:** นักข่าวต้องการข้อมูลสรุปเกี่ยวกับเหตุการณ์ 'การประชุมสุดยอด AI' โดยต้องการทราบว่าจัดขึ้นที่ประเทศใด (สมมติว่าจัดขึ้นที่ฝรั่งเศส), เมืองหลวงของประเทศนั้น, เวลาท้องถิ่น ณ ตอนนั้น, และข่าวล่าสุดเกี่ยวกับ 'AI' และ 'เศรษฐศาสตร์ (economics)' เพื่อหาความเชื่อมโยง

**งานของคุณ:**
1.  เขียนโค้ดที่รับคำสั่งจากผู้ใช้ เช่น `"การประชุมสุดยอด AI จัดขึ้นที่ไหน ช่วยบอกเมืองหลวง เวลาปัจจุบัน และสรุปข่าวล่าสุดเกี่ยวกับ AI และเศรษฐศาสตร์ให้ที"`
2.  LLM ควร (infer) ได้ว่าต้องหาเมืองหลวงของฝรั่งเศส และเรียกใช้ `get_capital_city`, `get_current_time`, และ `get_latest_news` (สองครั้งสำหรับสองหัวข้อ)
3.  **(ส่วนที่ท้าทาย)** ออกแบบ Pydantic Model ที่สามารถรวบรวม 'หัวข้อข่าว' หลายๆ หัวข้อได้แบบไดนามิก และสร้างข้อความสรุปเชิงวิเคราะห์สั้นๆ จากข้อมูลทั้งหมด

**Pydantic Model ที่คาดหวัง:**
```python
from pydantic import BaseModel, Field
from typing import Dict

class EventBriefing(BaseModel):
    event_name: str = "AI Summit"
    location_country: str
    location_capital: str
    local_time: str
    news_briefings: Dict[str, str] = Field(..., description="A dictionary mapping news topics to their summaries")
    analytic_summary: str # ข้อความสรุปที่สร้างขึ้นเอง เช่น 'The AI Summit in Paris is happening amidst news of powerful AI models, which may impact the economy.'
```
**คำใบ้:** การที่ LLM ต้องเรียก `get_latest_news` สองครั้งสำหรับสองหัวข้อที่แตกต่างกัน เป็นการทดสอบความสามารถในการจัดการ Tool Calls ที่ซ้ำซ้อนแต่มีพารามิเตอร์ต่างกัน คุณต้องมี Logic ในการรวบรวมผลลัพธ์จาก Tool Calls เหล่านี้ลงใน Dictionary ของ Pydantic

In [1]:
import IPython
import sys

def clean_notebook():
    IPython.display.clear_output(wait=True)
    print("Notebook cleaned.")

# Install necessary libraries
!pip install openai python-dotenv -q

# Clean up the notebook
clean_notebook()

Notebook cleaned.


In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [12]:
from pydantic import BaseModel, Field
from typing import Dict
import json

class NewsBriefing(BaseModel):
    topic: str =Field(description="หัวข้อของข่าว เช่น AI,Economics")
    detail: str =Field(description="เนื้อหาของข่าว")
    

class EventBriefing(BaseModel):
    event_name: str 
    location_country: str 
    location_capital: str 
    local_time: str 
    news_briefings: list[NewsBriefing]
    analytic_summary: str

print(EventBriefing.schema_json(indent=2))


{
  "$defs": {
    "NewsBriefing": {
      "properties": {
        "topic": {
          "description": "\u0e2b\u0e31\u0e27\u0e02\u0e49\u0e2d\u0e02\u0e2d\u0e07\u0e02\u0e48\u0e32\u0e27 \u0e40\u0e0a\u0e48\u0e19 AI,Economics",
          "title": "Topic",
          "type": "string"
        },
        "detail": {
          "description": "\u0e40\u0e19\u0e37\u0e49\u0e2d\u0e2b\u0e32\u0e02\u0e2d\u0e07\u0e02\u0e48\u0e32\u0e27",
          "title": "Detail",
          "type": "string"
        }
      },
      "required": [
        "topic",
        "detail"
      ],
      "title": "NewsBriefing",
      "type": "object"
    }
  },
  "properties": {
    "event_name": {
      "title": "Event Name",
      "type": "string"
    },
    "location_country": {
      "title": "Location Country",
      "type": "string"
    },
    "location_capital": {
      "title": "Location Capital",
      "type": "string"
    },
    "local_time": {
      "title": "Local Time",
      "type": "string"
    },
    "news_briefin

/var/folders/df/x56hm1c10td86gn10bv_zr240000gn/T/ipykernel_1378/1917992627.py:18: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  print(EventBriefing.schema_json(indent=2))


In [23]:
import datetime
import pytz
import json
import re

# Mock function
def get_latest_news(topic: str):
    """Gets the latest news on a topic."""
    news_data = {
        "AI": (
            "AI models are becoming increasingly powerful and are now being integrated "
            "into various industries including finance, healthcare, and education."
        ),
        "economics": (
            "Global economic trends are being influenced by rapid technological advancement, "
            "particularly in automation and artificial intelligence."
        ),
        "เศรษฐศาสตร์": (
            "เศรษฐกิจโลกได้รับผลกระทบจากการเปลี่ยนแปลงของเทคโนโลยี เช่น AI และระบบอัตโนมัติที่ลดความต้องการแรงงานมนุษย์."
        )
    }

    return news_data.get(
        topic,
        f"No latest news available for topic: {topic}"
    )

# Mock function
def get_capital_city(country):
    """Gets the capital city of a country."""
    capitals = {"France": "Paris"}
    return capitals.get(country, f"Capital of {country} not found.")

# Mock function
def get_current_time(timezone):
    """Gets the current time in a specific timezone."""
    try:
        tz = pytz.timezone(timezone)
        current_time = datetime.datetime.now(tz)
        return f"The current time in {timezone} is {current_time.strftime('%H:%M:%S')}."
    except pytz.UnknownTimeZoneError:
        return f"Unknown timezone: {timezone}"
    

def split_topics(topic_string):
    return [t.strip() for t in re.split(r"\s*(?:,|and|&)\s*", topic_string) if t.strip()]


In [24]:
all_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_latest_news",
            "description": "Gets the latest news on a topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string"}
                },
                "required": ["topic"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_capital_city",
            "description": "Get the capital city of a country.",
            "parameters": {
                "type": "object",
                "properties": {
                    "country": {"type": "string"}
                },
                "required": ["country"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current time in a specific timezone.",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {
                        "type": "string",
                        "description": "The timezone, e.g., 'France/Paris' or 'Asia/Tokyo'."
                    }
                },
                "required": ["timezone"]
            }
        }
    }
]




In [27]:
def execute_multi_function_call(user_query, tools):
    """A helper function to execute the full multi-function calling flow."""
    messages = [{"role": "system", "content": "สรุปเกี่ยวกับเหตุการณ์ 'การประชุมสุดยอด AI' สมมติว่าจัดขึ้นที่ฝรั่งเศส"},
        {"role": "user", "content": user_query}]
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    messages.append(response_message)  
    
    if response_message.tool_calls:
        available_functions = {
            "get_latest_news": get_latest_news,
            "get_capital_city": get_capital_city,
            "get_current_time": get_current_time,
        }
        
        
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions.get(function_name)
            
            if function_to_call:
                function_args = json.loads(tool_call.function.arguments)
                print(function_args)

                if function_name == "get_latest_news":
                    topic = function_args["topic"]
                    subtopics = split_topics(topic)

                    news_responses = []
                    for subtopic in subtopics:
                        result = get_latest_news(subtopic.strip())
                        news_responses.append(result.model_dump() if isinstance(result, BaseModel) else result)

                    # รวมผลลัพธ์ข่าวเป็น JSON list
                    combined_news = json.dumps(news_responses)

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": function_name,
                        "content": combined_news
                    })

                else:
                    function_response = function_to_call(**function_args)
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": function_name,
                        "content": json.dumps(function_response.model_dump() if isinstance(function_response, BaseModel) else function_response)
                    })


        print(messages)
        second_response = client.beta.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=messages,
        response_format=EventBriefing,
        )

        return second_response.choices[0].message.parsed
        
    else:
        return response_message.content
    


In [28]:
# user_query = input("กรอกข้อมูลการเดินทางของคุณ: ")
final_response = execute_multi_function_call("การประชุมสุดยอด AI จัดขึ้นที่ไหน ช่วยบอกเมืองหลวง เวลาปัจจุบัน และสรุปข่าวล่าสุดเกี่ยวกับ AI และเศรษฐศาสตร์ให้ที", all_tools)


{'country': 'ฝรั่งเศส'}
{'timezone': 'Europe/Paris'}
{'topic': 'AI'}
{'topic': 'เศรษฐศาสตร์'}
[{'role': 'system', 'content': "สรุปเกี่ยวกับเหตุการณ์ 'การประชุมสุดยอด AI' สมมติว่าจัดขึ้นที่ฝรั่งเศส"}, {'role': 'user', 'content': 'การประชุมสุดยอด AI จัดขึ้นที่ไหน ช่วยบอกเมืองหลวง เวลาปัจจุบัน และสรุปข่าวล่าสุดเกี่ยวกับ AI และเศรษฐศาสตร์ให้ที'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_AbmIcgR7JrV0fuOIKd54yeJy', function=Function(arguments='{"country": "ฝรั่งเศส"}', name='get_capital_city'), type='function'), ChatCompletionMessageToolCall(id='call_dgIF90tEYzcms2Iy62HjLCaV', function=Function(arguments='{"timezone": "Europe/Paris"}', name='get_current_time'), type='function'), ChatCompletionMessageToolCall(id='call_STjwLLAjSugqYPuJRAHipoZV', function=Function(arguments='{"topic": "AI"}', name='get_latest_news'), type='function'), ChatCompletionMessageToolCall(id='ca

In [29]:
print(final_response)

event_name='การประชุมสุดยอด AI' location_country='ฝรั่งเศส' location_capital='Paris' local_time='14:39:19' news_briefings=[NewsBriefing(topic='AI', detail='AI models are becoming increasingly powerful and are now being integrated into various industries including finance, healthcare, and education.'), NewsBriefing(topic='เศรษฐศาสตร์', detail='เศรษฐกิจโลกได้รับผลกระทบจากการเปลี่ยนแปลงของเทคโนโลยี เช่น AI และระบบอัตโนมัติที่ลดยังคงความต้องการมนุษย์.')] analytic_summary='การประชุมสุดยอด AI ในฝรั่งเศสครั้งนี้ มีการอภิปรายเกี่ยวกับบทบาทที่เพิ่มขึ้นของ AI ในการขับเคลื่อนนวัตกรรมและการเปลี่ยนแปลงในอุตสาหกรรมต่างๆ ควบคู่ไปกับการวิเคราะห์ผลกระทบทางเศรษฐกิจจากการนำเทคโนโลยีใหม่ ๆ มาใช้ ซึ่งอาจสร้างความท้าทายด้านแรงงานแต่เปิดโอกาสการเติบโตทางเศรษฐกิจในอนาคต'


In [31]:
for field, value in final_response.model_dump().items():
        if isinstance(value, list):
            print(f"{field}:")
            for idx, item in enumerate(value, 1):
                print(f"  topic no. {idx}:")
                for subfield, subvalue in item.items():
                    print(f"    {subfield}: {subvalue}")
        else:
            print(f"{field}: {value}")

event_name: การประชุมสุดยอด AI
location_country: ฝรั่งเศส
location_capital: Paris
local_time: 14:39:19
news_briefings:
  topic no. 1:
    topic: AI
    detail: AI models are becoming increasingly powerful and are now being integrated into various industries including finance, healthcare, and education.
  topic no. 2:
    topic: เศรษฐศาสตร์
    detail: เศรษฐกิจโลกได้รับผลกระทบจากการเปลี่ยนแปลงของเทคโนโลยี เช่น AI และระบบอัตโนมัติที่ลดยังคงความต้องการมนุษย์.
analytic_summary: การประชุมสุดยอด AI ในฝรั่งเศสครั้งนี้ มีการอภิปรายเกี่ยวกับบทบาทที่เพิ่มขึ้นของ AI ในการขับเคลื่อนนวัตกรรมและการเปลี่ยนแปลงในอุตสาหกรรมต่างๆ ควบคู่ไปกับการวิเคราะห์ผลกระทบทางเศรษฐกิจจากการนำเทคโนโลยีใหม่ ๆ มาใช้ ซึ่งอาจสร้างความท้าทายด้านแรงงานแต่เปิดโอกาสการเติบโตทางเศรษฐกิจในอนาคต
